In [0]:
import pyspark
import pyspark.sql.functions as F
import pyspark.sql.types as T
from functools import partial
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

In [0]:
def save_to_parquet(df, save_path):
    (df.write.format('parquet')
        .mode('overwrite')
        .option("header", "true")
        .save(save_path)
    )
    return print("successfully save parquet to ", save_path)

def save_to_csv(df, save_path):
    (df.coalesce(1)
        .write.format('csv')
        .mode('overwrite')
        .option("header", "true")
        .save(save_path)
    )
    return print("successfully save csv to ", save_path)

In [0]:
month_list = ['202604','202605','202606']
for month in month_list:
    volume_path = f"dbfs:/Volumes/int-cu-siampiwat/staging/report/report3/{month}/"
    path_mask = f'dbfs:/Volumes/int-cu-siampiwat/staging/report/report3/mask/{month}/'
    files = dbutils.fs.ls(volume_path)
    report1_files = [file.name for file in files if file.name.endswith('.csv/')]
    for file in report1_files:
        df = (spark.read
            .option("header", "true").option("inferSchema", "true")
            .csv(volume_path+file))

        df_marked = df.withColumn("hourly_unique_visitor_cnt", F.when(F.col("hourly_unique_visitor_cnt") <= 25, 25).otherwise(F.col("hourly_unique_visitor_cnt")))

        save_to_csv(df_marked, path_mask+file)
